# Notebook 1: Base Stack Export

Exports a single multi-band raster at MODIS scale in standard geographic coordinates (WGS84) per basin.
This asset contains ALL variables needed for downstream FRIP and GEDI analysis.

**Run once.** Downstream notebooks load this asset.

**Key design decisions (from GEE best practices + trial and error):**
- **Basin-Specific Geometry Clipping**: High-resolution datasets are filtered and clipped to the specific basin geometry (`basin_geom`) *before* executing `reduceResolution`. This bounds the reprojection grid and completely prevents `Reprojection output too large` errors.
- **Standard Geographic Reference (EPSG:4326)**: Standard WGS84 export projection resolves coordinate boundary transformation limits near the edges of the Amazon basin.
- **Independent `reduceResolution` chains**: Each band has its own independent aggregation chain — no cross-dataset dependencies.
- **Deferred Masking**: Masking thresholds are deferred to Stage 2 to preserve clean, continuous raster data.

In [ ]:
# =============================================================================
# BLOCK 1: SETUP AND CONFIGURATION
# =============================================================================
import ee

try:
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")

# Output destination
GEE_PROJECT = 'quantum-bonus-434714-t2'
ASSET_ROOT = f'projects/{GEE_PROJECT}/assets/DefaunationFromSpace'

# Study Regions — per-basin exports avoid spanning the Atlantic
CONGO_BBOX = ee.Geometry.Rectangle([8, -12, 35, 8])
AMAZON_BBOX = ee.Geometry.Rectangle([-73, -18, -44, 8])
BASINS = [('Congo', CONGO_BBOX), ('Amazon', AMAZON_BBOX)]

# Years
YEARS = list(range(2001, 2024))  # 2001-2023 inclusive

# GEE Dataset IDs
MODIS_NPP     = 'MODIS/061/MOD17A3HGF'
GLOFAS        = 'JRC/CEMS_GLOFAS/FloodHazard/v2_1'
MERIT_HYDRO   = 'MERIT/Hydro/v1_0_1'
FOREST_MASK   = 'projects/JRC/TMF/v1_2024/TransitionMap_MainClasses'
FOREST_CLASS  = 10   # Undisturbed since ~1982
SRTM          = 'USGS/SRTMGL1_003'
GEDI_L2B      = 'LARSE/GEDI/GEDI02_B_002_MONTHLY'
GEDI_L2A      = 'LARSE/GEDI/GEDI02_A_002_MONTHLY'
CHIRPS        = 'UCSB-CHG/CHIRPS/DAILY'
SOILGRIDS_CLAY = 'OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02'

# GEDI date range
GEDI_START = '2020-01-01'
GEDI_END   = '2023-12-31'

# CHIRPS date range (climatological mean)
PRECIP_START = '2001-01-01'
PRECIP_END   = '2023-12-31'

# --- MODIS NPP Reference Projection ---
_modis_col = ee.ImageCollection(MODIS_NPP).select('Npp')
MODIS_PROJ = ee.Image(_modis_col.first()).projection()
MODIS_SCALE = MODIS_PROJ.nominalScale()

print("\u2713 Configuration loaded.")
print(f"  Basins: {[b[0] for b in BASINS]}")
print(f"  Asset root: {ASSET_ROOT}")

In [ ]:
# =============================================================================
# BLOCK 2: BAND-BUILDING FUNCTIONS (GEOMETRY CLIP ENABLED)
# =============================================================================
# High-res layers are clipped to `basin_geom` at the very start of each function.
# This prevents GEE from evaluating massive global grids during reduceResolution.

def build_npp_bands(basin_geom):
    """MODIS NPP: median + 23 annual bands. Already at MODIS scale."""
    modis = ee.ImageCollection(MODIS_NPP).select('Npp').filterBounds(basin_geom)
    
    def get_annual(year):
        year = ee.Number(year)
        return modis.filter(
            ee.Filter.calendarRange(year, year, 'year')
        ).first().clip(basin_geom).set('year', year)
    
    annual_imgs = ee.ImageCollection.fromImages(
        ee.List(YEARS).map(get_annual)
    )
    
    median_npp = annual_imgs.median().clip(basin_geom).rename('Npp_median')
    annual_npp = annual_imgs.toBands().clip(basin_geom)
    band_names = [f'NPP_{y}' for y in YEARS]
    annual_npp = annual_npp.rename(band_names)
    
    return median_npp, annual_npp

def build_flood_frequency(basin_geom):
    """Flood frequency: count of return periods where flooding occurs."""
    glofas_col = ee.ImageCollection(GLOFAS).filterBounds(basin_geom)
    glofas = glofas_col.mosaic().clip(basin_geom)
    depth_bands = ['RP10_depth', 'RP20_depth', 'RP50_depth', 'RP75_depth',
                   'RP100_depth', 'RP200_depth', 'RP500_depth']
    
    flood_freq = glofas.select(depth_bands).gte(0).reduce(ee.Reducer.sum()).unmask()
    
    flood_proj = ee.Image(glofas_col.first()).projection()
    hnd_mask = ee.Image(MERIT_HYDRO).select('hnd').gt(0).clip(basin_geom)
    flood_freq = flood_freq.setDefaultProjection(crs=flood_proj).updateMask(hnd_mask)
    
    flood_reduced = flood_freq.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).rename('flood_freq')
    
    return flood_reduced

def build_forest_fraction(basin_geom):
    """JRC TMF forest fraction at MODIS resolution."""
    tmf_col = ee.ImageCollection(FOREST_MASK).filterBounds(basin_geom)
    tmf_proj = tmf_col.first().projection()
    
    forest = tmf_col.mosaic().eq(FOREST_CLASS).clip(basin_geom).setDefaultProjection(tmf_proj)
    
    forest_frac = forest.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).rename('forest_fraction')
    
    return forest_frac

def build_terrain(basin_geom):
    """Elevation and slope from SRTM, aggregated to MODIS resolution."""
    srtm = ee.Image(SRTM).clip(basin_geom)
    srtm_proj = srtm.select('elevation').projection()
    
    elev = srtm.select('elevation').setDefaultProjection(srtm_proj)
    elev_reduced = elev.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).rename('elevation')
    
    slp = ee.Terrain.slope(srtm).clip(basin_geom).setDefaultProjection(srtm_proj)
    slp_reduced = slp.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).rename('slope')
    
    return elev_reduced, slp_reduced

def build_hnd(basin_geom):
    """Height Above Nearest Drainage from MERIT Hydro."""
    hnd = ee.Image(MERIT_HYDRO).select('hnd').clip(basin_geom)
    hnd_proj = hnd.projection()
    
    hnd_reduced = hnd.setDefaultProjection(hnd_proj).reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).rename('hnd')
    
    return hnd_reduced

def build_gedi(basin_geom):
    """GEDI L2B: UOI, N; GEDI L2A: rh98 height."""
    gedi_b = ee.ImageCollection(GEDI_L2B).filterBounds(basin_geom).filterDate(GEDI_START, GEDI_END)
    native_b_proj = gedi_b.first().projection()
    
    def calc_uoi(img):
        pai = img.select('pai')
        pavd_z0 = img.select('pavd_z0')
        uoi = ee.Image(1).subtract(pavd_z0.divide(pai))
        return uoi.clamp(0, 1).rename('UOI')
    
    uoi_col = gedi_b.map(calc_uoi)
    
    uoi_mean = uoi_col.mean().clip(basin_geom).rename('GEDI_UOI').setDefaultProjection(native_b_proj)
    uoi_reduced = uoi_mean.reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535)
    
    uoi_count = uoi_col.count().clip(basin_geom).rename('GEDI_N').setDefaultProjection(native_b_proj)
    n_reduced = uoi_count.reduceResolution(
        reducer=ee.Reducer.sum(), maxPixels=65535)
    
    gedi_a = ee.ImageCollection(GEDI_L2A).filterBounds(basin_geom).filterDate(GEDI_START, GEDI_END)
    native_a_proj = gedi_a.first().projection()
    
    rh98_mean = gedi_a.select('rh98').mean().clip(basin_geom).rename('GEDI_rh98').setDefaultProjection(native_a_proj)
    rh98_reduced = rh98_mean.reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535)
    
    return uoi_reduced, n_reduced, rh98_reduced

def build_precip(basin_geom):
    """Mean annual precipitation from CHIRPS."""
    chirps = ee.ImageCollection(CHIRPS).filterBounds(basin_geom).filterDate(PRECIP_START, PRECIP_END)
    chirps_proj = chirps.first().projection()
    
    def annual_total(year):
        year = ee.Number(year)
        return chirps.filter(
            ee.Filter.calendarRange(year, year, 'year')
        ).sum().set('year', year)
    
    annual_precip = ee.ImageCollection.fromImages(
        ee.List(YEARS).map(annual_total)
    )
    mean_precip = annual_precip.mean().clip(basin_geom).rename('precip').setDefaultProjection(chirps_proj)
    
    return mean_precip

def build_clay(basin_geom):
    """Soil clay fraction from OpenLandMap/SoilGrids."""
    clay = ee.Image(SOILGRIDS_CLAY).clip(basin_geom)
    clay_mean = clay.reduce(ee.Reducer.mean()).rename('clay')
    clay_proj = clay.select(0).projection()
    
    clay_reduced = clay_mean.setDefaultProjection(clay_proj).reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    )
    
    return clay_reduced

print("\u2713 All geometry-clip functions defined.")

In [ ]:
# =============================================================================
# BLOCK 3: UNIT TESTS
# =============================================================================

def run_unit_tests():
    print("Running unit tests on individual band functions (using Congo BBox)...\n")
    passed = 0
    total = 9
    
    try:
        print(f"  [1/{total}] NPP bands...")
        npp_med, npp_ann = build_npp_bands(CONGO_BBOX)
        assert npp_med.bandNames().getInfo() == ['Npp_median']
        ann_bands = npp_ann.bandNames().getInfo()
        assert len(ann_bands) == 23, f"Expected 23, got {len(ann_bands)}"
        passed += 1
        print(f"    \u2713 Npp_median + {len(ann_bands)} annual bands")
    except Exception as e:
        print(f"    \u2717 FAILED: {e}")
    
    try:
        print(f"  [2/{total}] Flood frequency (GLOFAS v2_1)...")
        flood = build_flood_frequency(CONGO_BBOX)
        assert flood.bandNames().getInfo() == ['flood_freq']
        passed += 1
        print(f"    \u2713 flood_freq band")
    except Exception as e:
        print(f"    \u2717 FAILED: {e}")
    
    try:
        print(f"  [3/{total}] Forest fraction (JRC TMF)...")
        forest = build_forest_fraction(CONGO_BBOX)
        assert forest.bandNames().getInfo() == ['forest_fraction']
        passed += 1
        print(f"    \u2713 forest_fraction band")
    except Exception as e:
        print(f"    \u2717 FAILED: {e}")
    
    try:
        print(f"  [4/{total}] Elevation...")
        elev, slp = build_terrain(CONGO_BBOX)
        assert elev.bandNames().getInfo() == ['elevation']
        passed += 1
        print(f"    \u2713 elevation band")
    except Exception as e:
        print(f"    \u2717 FAILED: {e}")
    
    try:
        print(f"  [5/{total}] Slope...")
        assert slp.bandNames().getInfo() == ['slope']
        passed += 1
        print(f"    \u2713 slope band")
    except Exception as e:
        print(f"    \u2717 FAILED: {e}")
    
    try:
        print(f"  [6/{total}] HND (MERIT Hydro)...")
        hnd = build_hnd(CONGO_BBOX)
        assert hnd.bandNames().getInfo() == ['hnd']
        passed += 1
        print(f"    \u2713 hnd band")
    except Exception as e:
        print(f"    \u2717 FAILED: {e}")
    
    try:
        print(f"  [7/{total}] GEDI (UOI + N + rh98)...")
        uoi, n, rh98 = build_gedi(CONGO_BBOX)
        assert uoi.bandNames().getInfo() == ['GEDI_UOI']
        assert n.bandNames().getInfo() == ['GEDI_N']
        assert rh98.bandNames().getInfo() == ['GEDI_rh98']
        passed += 1
        print(f"    \u2713 GEDI_UOI, GEDI_N, GEDI_rh98 bands")
    except Exception as e:
        print(f"    \u2717 FAILED: {e}")
    
    try:
        print(f"  [8/{total}] Precipitation (CHIRPS)...")
        precip = build_precip(CONGO_BBOX)
        assert precip.bandNames().getInfo() == ['precip']
        passed += 1
        print(f"    \u2713 precip band")
    except Exception as e:
        print(f"    \u2717 FAILED: {e}")
    
    try:
        print(f"  [9/{total}] Clay (SoilGrids)...")
        clay = build_clay(CONGO_BBOX)
        assert clay.bandNames().getInfo() == ['clay']
        passed += 1
        print(f"    \u2713 clay band")
    except Exception as e:
        print(f"    \u2717 FAILED: {e}")
    
    print(f"\n{'='*60}")
    if passed == total:
        print(f"  \u2713 ALL {passed}/{total} TESTS PASSED")
    else:
        print(f"  \u2717 {passed}/{total} passed. Fix failures before proceeding.")
    print(f"{'='*60}")

run_unit_tests()

In [ ]:
# =============================================================================
# BLOCK 4: ASSEMBLE FULL STACK AND EXPORT
# =============================================================================

def build_full_stack(basin_geom):
    """Assembles all 34 bands into a single ee.Image clipped to the target geometry.
    
    Each band aggregated and clipped individually.
    """
    print("Assembling full stack...")
    npp_median, npp_annual = build_npp_bands(basin_geom)
    flood = build_flood_frequency(basin_geom)
    forest = build_forest_fraction(basin_geom)
    elev, slope = build_terrain(basin_geom)
    hnd = build_hnd(basin_geom)
    uoi, n, rh98 = build_gedi(basin_geom)
    precip = build_precip(basin_geom)
    clay = build_clay(basin_geom)
    
    stack = ee.Image.cat([
        npp_median,     # 1
        npp_annual,     # 2-24
        flood,          # 25
        forest,         # 26
        elev,           # 27
        slope,          # 28
        hnd,            # 29
        uoi,            # 30
        n,              # 31
        rh98,           # 32
        precip,         # 33
        clay,           # 34
    ]).toFloat()
    
    bands = stack.bandNames().getInfo()
    print(f"\u2713 Stack assembled: {len(bands)} bands")
    return stack

def safe_start(task, asset_id):
    try:
        ee.data.deleteAsset(asset_id)
        print(f"    Deleted existing: {asset_id.split('/')[-1]}")
    except Exception:
        pass
    task.start()

def export_base_stacks(dry_run=True):
    """Export the base stack as one asset per basin."""
    tasks = []
    
    for basin_name, basin_geom in BASINS:
        asset_id = f'{ASSET_ROOT}/BaseStack_{basin_name}'
        
        # Compile stack SPECIFICALLY clipped to this basin geometry
        stack = build_full_stack(basin_geom)
        
        task = ee.batch.Export.image.toAsset(
            image=stack,
            description=f'BaseStack_{basin_name}',
            assetId=asset_id,
            region=basin_geom,
            scale=MODIS_SCALE,
            crs='EPSG:4326',
            maxPixels=1e13
        )
        tasks.append((task, asset_id))
    
    print(f"\n\u2713 {len(tasks)} export tasks configured:")
    for _, aid in tasks:
        print(f"    {aid}")
    
    if dry_run:
        print("\nDRY RUN. Call export_base_stacks(dry_run=False) to start.")
    else:
        for task, asset_id in tasks:
            safe_start(task, asset_id)
            print(f"  \u2713 Started: {asset_id.split('/')[-1]}")
        print("\n\u2713 All tasks started!")
        print("  Monitor at: https://code.earthengine.google.com/tasks")

export_base_stacks(dry_run=True)